# Associate Refseq and Genebank gene names

This code reads a Refseq .gff file that contains both Refseq and Genebank gene names. Output is a list of Refseq gene names and the corresponding 
Genbank gene name.

The .gff file is read into a dictionary: Refseq gene name -> Genebank gene name.
1. If the line starts with "#", then skip to the next line.
2. If the line contains the the string "locus_tag=CPHY_RS[0-9]+", then match "CPHY_RS[0-9]+" and set it to the variable "refseq".
3. If the line contains the string "old_locus_tag=Cphy_[0-9]+", then match "Cphy_[0-9]+" and set it to the variable "genbank". Otherwise, set "genbnk" to "N/A".
4. Add a new key-value pair to the dictionary "gene_names" where the key is "refseq" and the value is "genbank".


## Setup

In [ ]:
# libraries
import re  # regular expressions

In [42]:
# File IO

file_gff = "/home/tolonen/Drives/Genoscope-googledrive/Lab_Projects/Genome_Cphy/Genome_Files/2025.11_Refseq/GCF_000018685.1_ASM1868v1/GCF_000018685.1_ASM1868v1_genomic.gff";
names_out = "/home/tolonen/Drives/Genoscope-googledrive/Lab_Projects/Genome_Cphy/Genome_Files/2025.11_Refseq/gene-names_Refseq_Genbank.tsv";


## Functions

In [43]:
def read_gff(file_path):
    """
    Reads a file line-by-line, extracts 'refseq' (locus_tag) and 
    'genbank' (old_locus_tag) identifiers, and maps them in a dictionary.
    
    Args:
        file_path (str): The path to the input gff file.
        
    Returns:
        dict: A dictionary mapping refseq IDs to GenBank IDs.
    """
    gff_names = {}  # initialize empty dictionary
    
    # Convert regex to pattern objects (using capturing groups)
    refseq_pattern = re.compile(r'locus_tag=(CPHY_RS\d+)')
    genbank_pattern = re.compile(r'old_locus_tag=(Cphy_\d+)')
    gene_line = re.compile(r"ID=gene-CPHY_RS\d+;")

    print(f"Processing file: {file_path}")

    try:
        with open(file_path, 'r') as file:
            for line in file:
                line = line.strip() # remove leading/trailing whitespace and newline
                
                # Skip lines starting with "#" or empty lines
                if line.startswith("#") or not line:
                    continue

                # skip lines that do not refer to "gene" features
                gene_line_match = gene_line.search(line)
                if not gene_line_match:
                    continue
                
                # Reset Refseq for each line 
                refseq = None
                
                # Check for locus_tag (refseq)
                refseq_match = refseq_pattern.search(line)
                if refseq_match:
                    refseq = refseq_match.group(1)
                    genbank = refseq_match.group(1) 
                
                # Check for old_locus_tag (genbank)
                genbank_match = genbank_pattern.search(line)
                if genbank_match:
                    genbank = genbank_match.group(1)
                else:
                    pass # move to next line of code

                if refseq:
                    gff_names[refseq] = genbank
                
    except FileNotFoundError:
        print(f"Error: Input file not found at {file_path}")
    except Exception as e:
        print(f"An unexpected error occurred during file processing: {e}")
        
    return gff_names

# Main

In [44]:
# make dictionary of refseq_name -> genebank_name
gff_names = read_gff(file_gff)

# count number of refseq names found
num_keys = len(gff_names)
print(f"Total refseq names found: {num_keys}")

# count number of genebank names found
target_string = "Cphy_"
count = 0

# Iterate through only the values of the dictionary
for genbank_name in gff_names.values():
    if target_string in genbank_name:
        count += 1

print(f"{count} Genbank genes are associated with Refseq names")

# print out file of Refseq and Genbank names
with open(names_out, 'w') as file_object:
    print("This file associates Refseq and Genbank names for C.phytofermentans genome using the GFF file for Refseq GCF_000018685.1_ASM1868v1. It was generated using gene-names_Refseq-Genbank.ipynb.\n", file = file_object)
    print("Refseq_name\tGenbank_name", file = file_object)

    for key, value in gff_names.items(): #items() loops through key-value pairs
        print(f"{key}\t{value}", file = file_object) # Print the key and the joined string on the same line, separated by a tab


Processing file: /home/tolonen/Drives/Genoscope-googledrive/Lab_Projects/Genome_Cphy/Genome_Files/2025.11_Refseq/GCF_000018685.1_ASM1868v1/GCF_000018685.1_ASM1868v1_genomic.gff
Total refseq names found: 4134
3884 Genbank genes are associated with Refseq names
